# ドイチのアルゴリズム（概要）

## オラクル
このアルゴリズムで使われるオラクルについて説明します。簡単に言うと、入力に対して0か1を返す関数です。
例として、何かを購入する場面を考えてみましょう。
入力が精肉店で買えるものであれば出力は1、買えないものであれば0とします。

つまり、

$$
f(x) = \begin{cases}
   1\ \  (x\ \text{は精肉店で買える})\\
    0\ \  (x\ \text{は精肉店で買えない})
  \end{cases}
$$

パソコンは精肉店では買えませんが、牛肉なら買えるので、

$$
f(\text{パソコン}) = 0,\ \ \ \ f(\text{牛肉}) = 1
$$

このように入力に応じて0か1を出力する関数をオラクルと呼びます。

量子コンピュータのアルゴリズムでは、入力状態 $x$ に対して特定の状態だけを答えとして取り出すためにオラクルを用います。
言い換えると、特定の状態のみを抽出するゲートです。

## ドイチのアルゴリズム
まず、ドイチが考案したアルゴリズムについて説明します。
ドイチのアルゴリズムは、$\lvert 0\rangle$ または $\lvert 1\rangle$ という1量子ビットの入力に対するオラクルを判定します。

$x =\lvert 0\rangle$ または $\lvert1\rangle$ のとき、考えられるオラクルは4通りあります。

$$
f(x) = \begin{cases}
   0\ \  (x= \lvert 0\rangle)\\
    0\ \  (x= \lvert1\rangle)
  \end{cases}
  \mathrm{or}\
  \begin{cases}
   0\ \  (x=\lvert 0\rangle)\\
    1\ \  (x=\lvert 1\rangle)
  \end{cases}
    \mathrm{or}\
  \begin{cases}
   1\ \  (x=\lvert 0\rangle)\\
    0\ \  (x=\lvert 1\rangle)
  \end{cases}
    \mathrm{or}\
  \begin{cases}
   1\ \  (x=\lvert 0\rangle)\\
    1\ \  (x=\lvert 1\rangle)
  \end{cases}
$$

ドイチのアルゴリズムは、この $f$ が $\lvert 0\rangle$ と $\lvert 1\rangle$ で同じ値を出力するのか、それとも異なる値を出力するのかを判定してくれます。

具体的な回路を考えてみましょう。

<img src="img/016/016_3_new.png" width="40%">

$U_f$ がオラクルです。
$q_0$ レジスタでは入力値 $x$ がそのまま返され、$q_1$ レジスタでは入力値 $y$ と $f(x)$ の和を $\rm{mod} 2$ した $y\oplus f(x)$ が返されます。

それぞれの状態を確認してみましょう。

$$
|\psi_1\rangle = H|0\rangle \otimes H|1\rangle = \frac{1}{\sqrt{2}}(\lvert0\rangle + \lvert1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert0\rangle - \lvert1\rangle)
$$

これを展開すると

$$
|\psi_1\rangle = \frac{1}{2} \lvert0\rangle \otimes (\lvert0\rangle - \lvert1\rangle) + \frac{1}{2} \lvert1\rangle \otimes (\lvert0\rangle - \lvert1\rangle)\ \ \ \ \cdots (\star)
$$

1番目の量子ビットが反転($\lvert0\rangle \leftrightarrow \lvert1\rangle$)すると符号が変わることがわかります。
この状況で、上記の4通りの $f(x)$ について $\lvert\psi_2\rangle$ を考えます。

$f(x)$ は1番目の量子ビットに作用するので、

$$
f(x)=0 \to q_1 \text{ は変化しない}
$$

$$
f(x)=1 \to q_1 \text{ は反転する}
$$

したがって、式 $(\star)$ を見ると

$$
f(0)=f(1) \to \text{2つの項は同じ符号を持つ}
$$

$$
f(0)\neq f(1) \to \text{2つの項は異なる符号を持つ}
$$

すると、

$$
\begin {align}
\lvert \psi_2 \rangle &= \begin{cases}
\pm \frac{1}{2} \lvert 0 \rangle \otimes (\lvert 0\rangle - \lvert 1\rangle) \pm \frac{1}{2} \lvert 1 \rangle \otimes (\lvert 0\rangle - \lvert 1\rangle)  \ \  \ \ (f(0)= f(1)) \\
\pm \frac{1}{2} \lvert 0 \rangle \otimes (\lvert 0\rangle - \lvert 1\rangle) \mp \frac{1}{2} \lvert 1 \rangle \otimes (\lvert 0\rangle - \lvert 1\rangle)  \ \  \ \ (f(0)\neq f(1)) \end{cases} \\
&= \begin{cases}
\pm H\lvert 0 \rangle \otimes H \lvert 1 \rangle \ \  \ \ (f(0)= f(1)) \\
 \pm H\lvert 1 \rangle \otimes H \lvert 1 \rangle \ \  \ \ (f(0)\neq f(1))
\end{cases}
\end {align}
$$


最後に $q_0$ に $H$ ゲートを作用させます。
すると

$$
\lvert \psi_3 \rangle = \begin{cases}
\pm \lvert 0\rangle \otimes H\lvert 1 \rangle\ \  \ \ (f(0)= f(1)) \\
\pm \lvert 1\rangle \otimes H\lvert 1 \rangle\ \  \ \ (f(0)\neq f(1))
\end{cases}
$$


符号 $\pm$ は測定結果に影響しないため、0番目の量子ビットを測定すればオラクルを判定できます。

以上がドイチのアルゴリズムの説明です。

blueqatを使って実装してみましょう。

In [1]:
from blueqat import Circuit
import numpy as np
from collections import Counter as _Counter

# Compatibility note: the new blueqat SDK reports measurement bitstrings with
# qubit 0 as the right-most character (standard binary ordering), while this
# notebook (like the old blueqat 0.3.x) reads qubit 0 as the left-most
# character. Patch Circuit.run once so sampled bitstrings keep matching the
# qubit ordering used throughout this notebook's explanation.
if not getattr(Circuit, '_qubit0_leftmost_patched', False):
    _original_run = Circuit.run
    def _run_qubit0_leftmost(self, *args, **kwargs):
        result = _original_run(self, *args, **kwargs)
        if isinstance(result, _Counter):
            return _Counter({key[::-1]: count for key, count in result.items()})
        return result
    Circuit.run = _run_qubit0_leftmost
    Circuit._qubit0_leftmost_patched = True

$f(x)$ の入出力の組み合わせとして、4通りのオラクルを考えました。
今回はこの4通りすべてを実装し、乱数によってオラクルをランダムに選択します。

In [2]:
def oracle_00(c):
    c.i[:]
    
def oracle_01(c):
    c.cx[0, 1]
    
def oracle_10(c):
    c.x[0]
    c.cx[0, 1]
    c.x[0]
    
def oracle_11(c):
    c.x[1]
    
def oracle(c):
    f = np.random.rand()
    if f < 0.25:
        oracle_00(c)
        return "f(0) = 0, f(1) = 0"
    elif f < 0.5:
        oracle_01(c)
        return "f(0) = 0, f(1) = 1"
    elif f < 0.75:
        oracle_10(c)
        return "f(0) = 1, f(1) = 0"
    else:
        oracle_11(c)
        return "f(0) = 1, f(1) = 1"

ここでドイチのアルゴリズムを使うと、測定結果から確率1でオラクルを判定できます。

In [3]:
c = Circuit(2)
c.x[1].h[:]
oracle_str = oracle(c)
c.h[0].m[0]
res = c.run(shots = 128)
print(res)
print("Selected oracle:", oracle_str)

Counter({'10': 128})
Selected oracle: f(0) = 0, f(1) = 1


オラクルが $f(0) = f(1)$ の場合、測定結果はすべてのサンプルで '00' になります。
オラクルが $f(0) \neq f(1)$ の場合、測定結果はすべてのサンプルで '10' になります。